# 01 · Análisis Exploratorio (EDA) — LoanSight

Exploración del warehouse Lending Club para entender la cartera, el desbalanceo de la variable objetivo de clasificación (`is_default`) y la relación de las features con `int_rate`.

**Objetivos**
1. Dimensionar el dataset y la calidad de los datos.
2. Cuantificar el desbalanceo de clases (default vs. pagado).
3. Observar cómo el grado de riesgo se relaciona con tasa y default.

In [ ]:
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent / "backend"))

DB_PATH = Path.cwd().parent / "data" / "processed" / "loansight.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)
print("Warehouse:", DB_PATH.exists())


In [ ]:
df = con.execute('''
    SELECT f.loan_amnt, f.term_months, f.annual_inc, f.dti,
           m.emp_length_years, p.purpose, g.grade,
           f.int_rate, f.loan_status, f.is_default
    FROM FACT_LOANS f
    JOIN DIM_PROPOSITO p ON f.proposito_sk = p.proposito_sk
    JOIN DIM_EMPLEO    m ON f.empleo_sk    = m.empleo_sk
    JOIN DIM_GRADO     g ON f.grado_sk     = g.grado_sk
''').df()
print(df.shape)
df.head()


## 1. Variable objetivo de clasificación y desbalanceo
Solo los préstamos *terminados* (Fully Paid / Charged Off) tienen etiqueta de default.

In [ ]:
status = df['loan_status'].value_counts()
print(status, '\n')
completed = df.dropna(subset=['is_default'])
rate = completed['is_default'].mean()
print(f'Préstamos terminados: {len(completed):,}')
print(f'Tasa de default: {rate:.2%}')

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
completed['is_default'].map({0:'Fully Paid',1:'Charged Off'}).value_counts().plot.bar(
    ax=ax, color=['#3fb950','#f85149'])
ax.set_title('Desbalanceo de clases (~20% default)')
ax.set_ylabel('N.º préstamos'); plt.tight_layout(); plt.show()

## 2. Tasa de interés y default por grado de riesgo
El grado (A–G) es asignado por Lending Club y resume el riesgo; esperamos que tanto `int_rate` como la tasa de default crezcan de A a G.

In [ ]:
by_grade = completed.groupby('grade').agg(
    n=('is_default','size'),
    default_rate=('is_default','mean'),
    avg_int_rate=('int_rate','mean')).reset_index()
by_grade

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(11,4))
axes[0].bar(by_grade['grade'], by_grade['default_rate']*100, color='#f85149')
axes[0].set_title('Tasa de default por grado'); axes[0].set_ylabel('%')
axes[1].bar(by_grade['grade'], by_grade['avg_int_rate'], color='#2f81f7')
axes[1].set_title('Tasa de interés media por grado'); axes[1].set_ylabel('int_rate %')
plt.tight_layout(); plt.show()

## 3. Distribución de `int_rate` (objetivo de regresión)

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.hist(df['int_rate'].dropna(), bins=50, color='#2f81f7', alpha=0.8)
ax.set_title('Distribución de int_rate'); ax.set_xlabel('int_rate (%)')
plt.tight_layout(); plt.show()
df['int_rate'].describe()

## 4. Correlación entre features numéricas e `int_rate`

In [ ]:
num = ['loan_amnt','term_months','annual_inc','dti','emp_length_years','int_rate']
corr = df[num].corr()
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(num))); ax.set_xticklabels(num, rotation=45, ha='right')
ax.set_yticks(range(len(num))); ax.set_yticklabels(num)
fig.colorbar(im); ax.set_title('Matriz de correlación'); plt.tight_layout(); plt.show()

## 5. Distribución por propósito del préstamo

In [ ]:
df['purpose'].value_counts().plot.barh(figsize=(7,5), color='#2f81f7')
plt.title('Préstamos por propósito'); plt.tight_layout(); plt.show()

## Conclusiones del EDA
- El dataset tiene ~2.26M préstamos; ~1.34M están terminados y son usables para clasificación.
- **Desbalanceo ~20%** de default → usaremos AUC-ROC, F1, precision y recall (no solo accuracy) y `class_weight='balanced'`.
- El **grado** explica fuertemente tanto `int_rate` como el default, pero NO está disponible en el contrato de la API; los modelos de producción usan solo features conocidas al momento de la solicitud.
- `debt_consolidation` y `credit_card` dominan los propósitos.